####hudson_tickets_python
# 🧠 Hudson Ticketing Python
##### **Author:** Barry Petersen  
##### **Email:** bthomasp@gmail.com  
##### **Date:** March 2026
##### **Environment:** Python / pandas
# ✅ Overview
###### This notebook runs analysis on hudson ticketing using 
### PYTHON

In [1]:
# ---------- Imports ----------
import pandas as pd
import numpy as np

ticket_df = spark.table("silver_hudson_ticket_events").limit(10000).toPandas()
customer_df = spark.table("silver_hudson_dim_customer").limit(10000).toPandas()

StatementMeta(, bdf9350a-1ecf-4142-b607-2eea4f771ec9, 3, Finished, Available, Finished, False)

In [2]:
#  Retrieve necessary columns
ticket_df = ticket_df[["order_id", "customer_id", "amount", "sales_date"]].copy()
customer_df = customer_df[["customer_id", "segment"]].copy()

#  Clean nulls+
ticket_df["amount"] = ticket_df["amount"].fillna(0).astype(float)
customer_df["segment"] = customer_df["segment"].fillna("Unknown")

#  Drop unused
ticket_df = ticket_df.dropna(subset=["order_id", "customer_id"])

StatementMeta(, bdf9350a-1ecf-4142-b607-2eea4f771ec9, 4, Finished, Available, Finished, False)

In [3]:
#  Join
merged_df = ticket_df.merge(customer_df, on="customer_id", how="left")

#  Handle agg class
merged_df["segment"] = merged_df["segment"].fillna("Unknown")

StatementMeta(, bdf9350a-1ecf-4142-b607-2eea4f771ec9, 5, Finished, Available, Finished, False)

In [4]:

segment_perf = (
    merged_df.groupby("segment", as_index=False)
        .agg(
            total_revenue=("amount", "sum"),
            total_orders=("order_id", "nunique")
            )
)

segment_perf["revenue_per_order"] = np.where(
    segment_perf["total_orders"] > 0,
    (segment_perf["total_revenue"] / segment_perf["total_orders"]).round(2),
    0
)

segment_perf = segment_perf.sort_values("revenue_per_order", ascending=False)

print(segment_perf)

StatementMeta(, bdf9350a-1ecf-4142-b607-2eea4f771ec9, 6, Finished, Available, Finished, False)

   segment  total_revenue  total_orders  revenue_per_order
3  Tourist      140136.18          1019             137.52
5      VIP      139547.50          1019             136.95
4  Unknown      135985.40           997             136.39
0    Group      128233.64           952             134.70
2  Student      129816.97           971             133.69
1    Local      138935.73          1042             133.34
